In [2]:
import boto3
import json
import time
import pandas as pd
import argparse 
import random

# session = boto3.Session(profile_name='corp-us-east-1')
session = boto3.Session(profile_name='default')


# Define the table name
table_name = 'prompt_hub_table'
# Create DynamoDB resource
dynamodb = session.resource('dynamodb')
table = dynamodb.Table(table_name)

In [3]:

def filter_items(field,val):
    # Scan the table to get all items where is_external is true
    response = table.scan(
        FilterExpression=f'{field} = :val',
        ExpressionAttributeValues={':val': val}
    )
    
    items = response['Items']
    print(f"found:{len(items)}")
    
    # Handle pagination if there are more items
    while 'LastEvaluatedKey' in response:
        response = table.scan(
            FilterExpression='{field} = :val',
            ExpressionAttributeValues={':val': val},
            ExclusiveStartKey=response['LastEvaluatedKey']
        )
        items.extend(response['Items'])
    return items

def update_items(items,field,val):
    # Update each item's delete_status
    updated_count = 0
    for item in items:
        # Get the primary key values from your item
        # Modify these according to your table's primary key structure
        key = {
            'id': item['id']  # Assuming 'id' is your primary key
            # Add other key attributes if you have a composite key
        }

        # Update the item
        table.update_item(
            Key=key,
            UpdateExpression=f'SET {field} = :val',
            ExpressionAttributeValues={
                ':val': val
            }
        )
        updated_count += 1

    print(f"Successfully updated {updated_count} items")
    return updated_count

In [4]:
# 把is_external = true都删除
items = filter_items("is_external",True)
update_items(items, "delete_status","deleted")

found:22
Successfully updated 22 items


22

In [5]:
# 把is_recommended = true 改成 is_recommended = False
# items = filter_items("is_recommended",False)
# update_items(items, "is_recommended",True)

## upload new template

In [6]:
def upload_to_dynamodb(table_name, json_data):
    dynamodb = session.resource('dynamodb')
    table = dynamodb.Table(table_name)

    for item in json_data:
        table.put_item(Item=item)

    print(f"Data uploaded to DynamoDB table: {table_name}")


def generate_id():
    timestamp = int(time.time() * 1000)  # Get the current timestamp in milliseconds
    random_number = str(random.randint(0, 16**6))  # Generate a random 6-digit number
    return f"{timestamp}-{random_number}"


def process_excel(filename):
    df = pd.read_excel(filename)
    time_tuple = time.localtime( time.time())
    createtime = time.strftime("%Y-%m-%d %H:%M:%S", time_tuple)
    df.drop(['Cx Cases in Prod','Team'],inplace=True,axis=1)
    df.rename(columns={'Category':'category',
                       'Name':'demo_name',
                       'Description':'description',
                       'Further Support':'further_support',
                       'Status':'demo_type',
                       'Simple Demo Introduction Deck':'deck_link',
                       'Demo Video Link':'demo_link',
                       'Code Repo Link':'code_repo_link',
                       'Contact':'contact'
                       }, inplace=True)
    df['createtime'] = createtime
    df['company'] = 'default'
    df['template'] = ''
    df['id'] = df.apply(lambda x: generate_id(), axis=1)
    df['demo_version'] = '2025v1'
    # df['industry'] = df.apply(lambda x: [{"label":i,"value":i}  for i in x['industry'].split('|') ],axis=1)
    
    df_dict = json.loads(df.to_json(orient='index'))
    return  list(df_dict.values())


In [ ]:


filename = "demo_hub_v2025.xlsx"

json_data = process_excel(filename)
print(json_data)

# Upload the JSON data to a new DynamoDB table
upload_to_dynamodb(table_name, json_data)
print(f'uploded data from {filename}')